# Phase 0 walkthrough: SimCLR + frozen-encoder probe on (synthetic) X-ray images

This notebook reproduces the **Phase 0 pipeline** (`src/simclr_hpl/cli/xray_screening.py`)
end-to-end on a tiny, made-up dataset — a handful of plain red and blue PNGs — so it runs
in well under a minute on a CPU with no downloads.

It is **not** meant to produce a meaningful model (12 solid-color images per class won't
teach a network anything useful about X-ray threats!). It exists purely so you can *watch
every stage of the real pipeline run*, end to end, and connect each step back to the
actual library code:

1. Build a tiny synthetic binary image dataset (`positive/` = red, `negative/` = blue).
2. Load it with `load_binary_image_records` (the same loader the CLI uses).
3. Build a `ResNet18Encoder` via `build_encoder("resnet18", ...)` — the backbone that
   replaces the small MNIST `Encoder` for RGB X-ray images.
4. Run **one epoch of SimCLR pretraining** (`pretrain_simclr` + `NTXentLoss`) on
   unlabeled augmented views (`ContrastiveViewDataset` + `build_rgb_simclr_transform`).
5. **Freeze** the encoder and train a `LinearProbe` for a couple of epochs
   (`train_classifier`), then measure accuracy with `evaluate_classifier`.

For the conceptual "why" behind each step, see the companion explainer:
`docs/explainers/phase0-xray-baseline.md`.


In [1]:
import tempfile
from pathlib import Path

from PIL import Image

# Make a throwaway directory that looks like the folder layout
# `load_binary_image_records` expects: <root>/<class_name>/*.png
tmp_dir = tempfile.mkdtemp(prefix="phase0_xray_demo_")
data_root = Path(tmp_dir) / "toy_xray"

N_PER_CLASS = 12
IMAGE_SIZE = 32  # tiny on purpose -- keeps everything fast on CPU

# "positive" = solid red squares (stands in for "threat"), "negative" = solid blue
# squares (stands in for "no-threat"). Real X-ray images obviously look nothing like
# this -- this is just enough structure for the pipeline's plumbing to run end to end.
class_colors = {"positive": (200, 30, 30), "negative": (30, 30, 200)}

for class_name, color in class_colors.items():
    class_dir = data_root / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    for i in range(N_PER_CLASS):
        Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=color).save(class_dir / f"{i:02d}.png")

print(f"Synthetic dataset written to: {data_root}")
for class_name in class_colors:
    n_files = len(list((data_root / class_name).glob("*.png")))
    print(f"  {class_name}/: {n_files} images")


Synthetic dataset written to: /var/folders/hs/jhbbsd052zz044qtbzkkhxl80000gn/T/phase0_xray_demo_jx9hhlnt/toy_xray
  positive/: 12 images
  negative/: 12 images


## Step 1 — Load the records with `load_binary_image_records`

This is the exact same loader the `xray-screening` CLI uses
(`src/simclr_hpl/data.py:100-128`, wired in at `cli/xray_screening.py:64`). It scans the
*immediate subfolders* of `root`, maps each subfolder name to a 0/1 label via the
`class_to_label` dict you provide, and returns a flat list of
`{"path":..., "label":..., "class_name":...}` records. Folder names not present in the
mapping are silently skipped — that's what makes it tolerant of slightly different real
dataset layouts.

Here `positive -> 1` ("threat") and `negative -> 0` ("no-threat"), matching
`configs/xray_screening.yaml`'s `data.class_to_label`.


In [2]:
from collections import Counter

from simclr_hpl.data import load_binary_image_records

records = load_binary_image_records(data_root, class_to_label={"positive": 1, "negative": 0})

label_counts = Counter(int(r["label"]) for r in records)
print(f"Loaded {len(records)} records")
print(f"Label counts (1 = threat/positive, 0 = no-threat/negative): {dict(label_counts)}")
print("Example record:", {k: str(v) for k, v in records[0].items()})


Loaded 24 records
Label counts (1 = threat/positive, 0 = no-threat/negative): {0: 12, 1: 12}
Example record: {'path': '/var/folders/hs/jhbbsd052zz044qtbzkkhxl80000gn/T/phase0_xray_demo_jx9hhlnt/toy_xray/negative/00.png', 'label': '0', 'class_name': 'negative'}


## Step 2 — Build the `ResNet18Encoder` via `build_encoder`

The original `Encoder` (`src/simclr_hpl/models.py:8-41`) is a small hand-rolled CNN sized
for 28x28 grayscale MNIST digits (`output_dim = 4096`, `input_channels=1` by default).
X-ray images are larger and RGB, so Phase 0 swaps in `ResNet18Encoder`
(`models.py:44-58`), which wraps `torchvision.models.resnet18` and replaces its final
classification layer with `nn.Identity()` — turning a 1000-way ImageNet classifier into a
plain **512-dimensional feature extractor** (`output_dim = 512`).

`build_encoder(name, input_channels, pretrained)` (`models.py:61-71`) is the factory the
CLI uses to pick between `"small"` and `"resnet18"` based on the YAML config. We call it
exactly the way `cli/xray_screening.py:89-93` does, with `input_channels=3` for RGB.


In [3]:
import torch

from simclr_hpl.models import build_encoder

torch.manual_seed(0)

encoder = build_encoder("resnet18", input_channels=3, pretrained=False)
print(type(encoder).__name__, "output_dim =", encoder.output_dim)

# Sanity check: feed a batch of random RGB images through and look at the feature shape.
dummy_batch = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    features = encoder(dummy_batch)
print("Feature batch shape:", tuple(features.shape), "  (batch_size, output_dim)")


ResNet18Encoder output_dim = 512
Feature batch shape: (2, 512)   (batch_size, output_dim)


## Step 3 — One epoch of SimCLR pretraining (no labels used!)

`ContrastiveViewDataset` (`data.py:31-41`) wraps an image dataset and, for each item,
ignores the label and returns **two independently-augmented views of the same image**:
`(transform(image), transform(image))`. The transform is `build_rgb_simclr_transform`
(`data.py:175-194`) — resize, random-resized-crop, flips, color jitter, grayscale —
applied stochastically so the two views differ.

`NTXentLoss` (`training.py:12-36`) then pulls each pair of views together (in cosine-
similarity space, scaled by a `temperature`) while pushing every other image in the batch
apart. `pretrain_simclr` (`training.py:39-73`) is the loop that runs both views through
the encoder + a small `ProjectionHead`, computes that loss, and steps the optimizer.

We use a tiny `image_size=32`, a small batch, and a single epoch — enough to *exercise*
every line of the real pretraining path, not to learn anything meaningful from 24 solid-
color images.


In [4]:
from torch.utils.data import DataLoader

from simclr_hpl.data import (
    ContrastiveViewDataset,
    ImagePathDataset,
    build_rgb_simclr_transform,
)
from simclr_hpl.models import ProjectionHead
from simclr_hpl.training import NTXentLoss, pretrain_simclr

device = torch.device("cpu")
image_size = 32

simclr_transform = build_rgb_simclr_transform(image_size=image_size)

# ImagePathDataset(records, transform=None) loads raw PIL images; ContrastiveViewDataset
# wraps it and applies `simclr_transform` twice per item -- this is exactly the
# `contrastive` / `contrastive_loader` setup in cli/xray_screening.py:78-87.
contrastive_dataset = ContrastiveViewDataset(
    ImagePathDataset(records, transform=None), simclr_transform
)
contrastive_loader = DataLoader(
    contrastive_dataset, batch_size=4, shuffle=True, drop_last=True, num_workers=0
)

projection_head = ProjectionHead(input_dim=encoder.output_dim, output_dim=32)
criterion = NTXentLoss(temperature=0.5)
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(projection_head.parameters()), lr=1e-3
)

history = pretrain_simclr(
    encoder=encoder,
    projection_head=projection_head,
    data_loader=contrastive_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=1,
)
print("SimCLR pretraining loss (per epoch):", history["loss"])


SimCLR pretraining:   0%|          | 0/1 [00:00<?, ?it/s]

SimCLR pretraining loss (per epoch): [1.6183873812357585]


## Step 4 — Freeze the encoder, train a `LinearProbe`, evaluate

Now comes the **evaluation** stage of Phase 0: we `freeze_module(encoder)`
(`training.py:172-174`, sets `requires_grad = False` on every encoder parameter) so the
SimCLR-pretrained weights stay fixed, then bolt on a `LinearProbe`
(`models.py:87-94` — just one `nn.Linear(encoder.output_dim, num_classes)`). If this
single linear layer can separate the classes well using only the *frozen* features,
that's evidence the representation itself is doing the work.

We train it for 2 tiny epochs with `train_classifier` (`training.py:130-169`, ordinary
supervised cross-entropy over the small labeled subset) and then measure held-out
accuracy with `evaluate_classifier` (`training.py:76-98`) — the same function that
produces the `accuracy` number written to `metrics.json` by the real CLI
(`cli/xray_screening.py:144-148`).

Because our synthetic images are trivially separable solid colors, don't be surprised if
the probe gets ~100% — that's the synthetic data being *too* easy, not a meaningful result
about representation quality (on real X-ray data this number is the actual signal to
watch).


In [5]:
from torch import nn

from simclr_hpl.data import (
    build_rgb_normalize_transform,
    build_train_val_subsets,
    split_records_stratified,
)
from simclr_hpl.models import LinearProbe
from simclr_hpl.training import (
    evaluate_classifier,
    freeze_module,
    train_classifier,
    trainable_parameters,
)

# Same stratified split the CLI uses (cli/xray_screening.py:65-67), just with a larger
# test_size so our tiny 24-image dataset still leaves enough for train/val/test subsets.
train_records, test_records = split_records_stratified(records, test_size=0.34, seed=0)

eval_transform = build_rgb_normalize_transform(image_size=image_size)
eval_train_dataset = ImagePathDataset(train_records, transform=eval_transform)
eval_test_dataset = ImagePathDataset(test_records, transform=eval_transform)

probe_train, probe_val = build_train_val_subsets(eval_train_dataset, validation_size=0.34, seed=0)
# NOTE: batch_size=3 is chosen so every batch (incl. the last one) has more than
# one sample -- ResNet18's BatchNorm layers raise an error on batch size 1 while
# in training mode (`model.train()` puts the whole probe, including the frozen
# encoder submodule, into training mode). With 9 train / 6 val / 9 test images
# this divides evenly.
train_loader = DataLoader(probe_train, batch_size=3, shuffle=True)
val_loader = DataLoader(probe_val, batch_size=3)
test_loader = DataLoader(eval_test_dataset, batch_size=3)

# --- Freeze the SimCLR-pretrained encoder; only the new linear head will learn. ---
freeze_module(encoder)
probe = LinearProbe(encoder, num_classes=2).to(device)

probe_optimizer = torch.optim.Adam(trainable_parameters(probe), lr=1e-3)
probe_criterion = nn.CrossEntropyLoss()

probe_history = train_classifier(
    model=probe,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=probe_optimizer,
    criterion=probe_criterion,
    device=device,
    epochs=2,
)
print("Probe train/val accuracy per epoch:")
print("  train_accuracy:", probe_history["train_accuracy"])
print("  val_accuracy:  ", probe_history["val_accuracy"])

test_metrics = evaluate_classifier(probe, test_loader, probe_criterion, device)
print(f"\nHeld-out test metrics -> loss: {test_metrics['loss']:.4f}, accuracy: {test_metrics['accuracy']:.4f}")


Supervised training:   0%|          | 0/2 [00:00<?, ?it/s]

Probe train/val accuracy per epoch:
  train_accuracy: [0.4444444444444444, 0.7777777777777778]
  val_accuracy:   [0.5, 0.5]

Held-out test metrics -> loss: 0.7883, accuracy: 0.4444


## Takeaways — and how this maps onto the real `xray-screening` CLI

What you just watched, end to end, is *exactly* the pipeline that
`src/simclr_hpl/cli/xray_screening.py::main` runs on real X-ray data — just with tiny
synthetic images, smaller batches/epochs, and `image_size=32` instead of `224`:

| This notebook | Real CLI (`cli/xray_screening.py`) |
|---|---|
| 24 solid-color PNGs in `positive/`/`negative/` | Real X-ray images under `data/sixray/...` (`load_binary_image_records`, line 64) |
| `build_encoder("resnet18", input_channels=3)` | Same factory call, driven by `configs/xray_screening.yaml`'s `model.encoder` (lines 89-93) |
| 1 epoch of `pretrain_simclr` on `max_pretrain_images`-many (here: all 24) unlabeled images | `train_cfg["epochs"]` epochs (config: 20) on up to `max_pretrain_images` (config: 4000) unlabeled images — the **large, label-free** pool (lines 75-87, 103-112) |
| `freeze_module` + `LinearProbe`, trained on a stratified subset | Same, but on `subsample_per_class(..., max_labeled_per_class)` — a deliberately **small, balanced labeled** budget (config: 500/class) (lines 114-145) |
| `evaluate_classifier` -> one accuracy number | Same, written into `artifacts/xray_screening/metrics.json` for both `linear_probe` and `mlp_probe` (lines 144-148) |

The whole point of this design (see `docs/explainers/phase0-xray-baseline.md`, sections 2
and 4) is the **asymmetry** between the two sample budgets: SimCLR gets to look at *lots*
of unlabeled images, while the probe — the only stage that needs ground-truth labels — is
trained on relatively few. If the probe still reaches strong accuracy, that's the
"low-label" thesis working: good representations learned without labels mean you need
far fewer labels to build an accurate classifier on top.

Finally, recall the through-line to deployment: a single accuracy number says whether the
model is *good*; `business.compute_review_queue_metrics` (`src/simclr_hpl/business.py`)
is the next layer, which uses **per-prediction confidence** and an
`auto_decision_threshold` to decide whether to trust the model automatically or route the
image to a human reviewer — the same confidence-gated principle that will drive the later
temporal-screening demo.

To run the real pipeline (no synthetic data) yourself:

```bash
uv run pytest tests/test_xray_cli.py -v          # synthetic smoke test, no data needed
uv run xray-screening --config configs/xray_screening.yaml   # real run, needs data/sixray
```
